## Workspace setup

In [ ]:
from datetime import datetime  
import uproot
from functools import partial
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

import importlib

# Notebooks run from VSCode use home directory as a base path
# while notebooks run from JupyterLab use the current directory as a base path
import sys
sys.path.append("/home/akalinow/scratch/ELITPC/TPCReco/PythonAnalysis/python/")

# Change directory to the working directory
import os
os.chdir('/scratch_hdd/akalinow/ELITPC/PythonAnalysis/')

## Training dataset preparation

In [ ]:
import io_functions as io
importlib.reload(io)

import plotting_functions as plf
importlib.reload(plf)

batchSize = 32
dataset = tf.data.Dataset.load('MergedEvent_Track3D_TwoProng_gun_MC', compression="GZIP")
dataset = dataset.batch(batchSize, drop_remainder=True)
dataset = dataset.map(lambda x: x['sim'])
dataset = dataset.map(lambda x,y: (tf.reshape(x, (-1,)+io.projections.shape), tf.reshape(y, (-1,9))))

dataset = dataset.map(lambda x,y: (x,y[:,::3])) #vertex only

#set endpoints Z position wrt vertex. Set vertex Z to zero, as this is arbitrary due to triggering principle
#dataset = dataset.map(lambda x,y: (x, y-y[:,-3:-2]*tf.constant([[0,0,0,0,0,0,1,1,1]],dtype=tf.float32)))

randomTranslation = tf.keras.layers.RandomTranslation(
    height_factor=0.0, width_factor=(0,0.3),
    fill_mode='constant', fill_value=0.0)

#dataset = dataset.map(lambda x,y: (randomTranslation(x), y))

dataset = dataset.prefetch(tf.data.AUTOTUNE)
#tfds.benchmark(dataset.take(1000),batch_size=batchSize)

In [ ]:
y = next(iter(dataset))[1]
#y = tf.reshape(y, (-1,3,3))
print(y[0])
#print(y[0,:,0])

## Model definition

In [ ]:
def getModel():

  model = tf.keras.Sequential([
  tf.keras.layers.Input(shape=(256,512,3), name="input_image", dtype=tf.float32),
  #tf.keras.layers.GaussianNoise(stddev=0.05, name='gauss_noise'),
  tf.keras.layers.Resizing(height=256, width=256), 
  tf.keras.layers.Conv2D(16, 4, padding='same', activation='relu', 
                          data_format="channels_last"),
  tf.keras.layers.MaxPooling2D(),
  tf.keras.layers.Conv2D(32, 2, padding='same', activation='relu', 
                          data_format="channels_last"),
  tf.keras.layers.MaxPooling2D(),
  tf.keras.layers.Conv2D(64, 2, padding='same', activation='relu', 
                          data_format="channels_last"),
  #tf.keras.layers.MaxPooling2D(),
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(3, activation="sigmoid"),
  tf.keras.layers.Rescaling(scale=128, name='output_rescale')
  ])

  initial_learning_rate = 0.01
  lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate,
                  decay_steps=836,
                  decay_rate=0.98,
                  staircase=False)

  optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule) 
  model.compile(optimizer = optimizer, 
                loss = 'mse', 
                metrics=['mse', 'mape']) 

  model.summary()
  return model
#########################################################
#########################################################  
def getModel1():

  model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(256,512,3), name="input_image", dtype=tf.float32),
    #tf.keras.layers.GaussianNoise(stddev=0.05, name='gauss_noise'),
    tf.keras.layers.Conv2D(3, kernel_size=(1,2), strides=(1,2), padding='valid', activation='relu', 
                          data_format="channels_last", groups=3),
    tf.keras.layers.Conv2D(32*3, 4, padding='same', activation='relu', 
                          data_format="channels_last", groups=3),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(32*3, 2, padding='same', activation='relu', 
                          data_format="channels_last", groups=3),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(32, 2, padding='same', activation='relu', 
                          data_format="channels_last"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
    tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
    tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
    tf.keras.layers.Dense(9, activation="linear"),
    #tf.keras.layers.Rescaling(scale=128, name='output_rescale')
  ])

  initial_learning_rate = 0.01
  lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate,
                  decay_steps=836,
                  decay_rate=0.98,
                  staircase=False)

  optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule) 
  model.compile(optimizer = optimizer, 
                loss = 'mse', 
                metrics=['mse', 'mape']) 

  model.summary()
  return model
##########################################################
##########################################################

In [ ]:
model = getModel()
#model = getModelWithTL()

#model_path = "./training/0100_2025_Oct_13_18_48_39.keras"
#model = tf.keras.models.load_model(model_path)
#model.evaluate(dataset.take(10))
#model.summary()

#item = next(iter(dataset.take(1)))
#model(item)[0]

## Model training

In [ ]:
%%time

import plotting_functions as plf
importlib.reload(plf)

log_dir = "logs/fit/" + datetime.now().strftime("%Y%m%d-%H%M%S")
#tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1, profile_batch=(10, 20))
early_stop_callback = tf.keras.callbacks.EarlyStopping(patience=5, verbose=1)
callbacks =  [early_stop_callback] 

epochs=10
model = getModel()
model.trainable = True

initial_learning_rate = 0.01
decay_steps = 2*836
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate,
                  decay_steps=decay_steps,
                  decay_rate=0.98,
                  staircase=False)

optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule) 
model.compile(optimizer = optimizer, 
                loss = 'mse', 
                metrics=['mse']) 

history = model.fit(dataset.skip(10), 
                        epochs=epochs,
                        verbose = 1,
                        validation_data = dataset.take(10),
                        #callbacks=callbacks
                        )
plf.plotTrainHistory(history)

current_time = datetime.now().strftime("%Y_%b_%d_%H_%M_%S")
print("Training start. Current Time =", current_time)

job_dir = f"training/{epochs:04d}_"+current_time+".keras"
model.save(job_dir)

item = next(iter(dataset))
model(item)[0]

## Model performance on training data.

Fill Pandas DataFrame with true and response values.

In [ ]:
%%time
import utility_functions as utils
importlib.reload(utils)

#model_path = "./training/0050_2025_Oct_14_15_14_48.keras"
#model = tf.keras.models.load_model(model_path)

df = utils.getEmptyPandasDataset(utils.columnsXYZ)

for aBatch in dataset.take(10_000): 
    df = utils.fillPandasDataset(aBatch, df, model)     
    
for aBatch in dataset.take(3):
    plf.plotEvent(aBatch, model=model)

df.describe()    

In [ ]:
import plotting_functions as plf
importlib.reload(plf)

import utility_functions as utils
importlib.reload(utils)

for aBatch in dataset.skip(100).take(3):
    plf.plotEvent(aBatch, model=model)

### Resolution plots

In [ ]:
import plotting_functions as plf
importlib.reload(plf)

#plf.controlPlots(df)
plf.plotEndPointRes(df=df, edge="Vtx",coordinates=["x", "y", "z"])
plf.plotEndPointRes(df=df, edge="Alpha",coordinates=["x", "y", "z"])
plf.plotEndPointRes(df=df, edge="Carbon",coordinates=["x", "y", "z"])

plf.plotLengthPull(df, partName="Alpha")
plf.plotLengthPull(df, partName="Carbon")
plf.plotLengthPullEvolution(df)
plf.plotOpeningAngleCos(df)